# Five paths, one scoreboard -- but which recommender is *best*?

Part 1 is nearly built. We have five retrieval paths -- a random floor, popularity (Unit 2),
lexical keyword search (Unit 3), item-item collaborative filtering (Unit 4), matrix factorization
(Unit 5) -- and a frozen scoreboard that scores each one by **hit@10** on the held-out `val` split.
The last scoreboard printed a tidy ladder: popularity about 0.108, lexical 0.158, item-item CF
0.252, MF 0.276. Case closed, ship MF?

Not yet. That ladder answers only one question -- *how often does the right book land somewhere in
the top 10?* -- and it hides two others that a real recommender lives or dies by. **Which path is
best, and best at *what*?** MF wins on hit@10, but it recommends the same narrow sliver of the
catalog to almost everyone; item-item CF reaches about 6.6x more books. And if we could *combine*
the paths, would the whole beat the best part?

This unit deepens how we *measure* a recommender -- ranking-aware metrics (precision@k, NDCG) and
**beyond-accuracy** metrics (coverage, diversity, novelty) -- and then builds the book's **first
blended recommender**: calibrate every path to a common scale, take a weighted union, and rank. We
will measure it honestly on `val`, and the result carries a genuine surprise: the blend beats the
best single path *and* covers far more catalog -- yet one of its four paths is pulling it *down*.

## Load the data and fit the Part-1 paths

The Unit 1-5 substrate is unchanged. `generated_dir()` locates the seeded data slice (run the
recsys generators first if it complains); `load_catalog` gives the catalog, the interaction log is
the gzip'd CSV of `reader_id, item_id, split, label` rows, and `cold_partitions.json` lists the
`cold_readers` held out of every score. We fit all four learned paths **once** here -- the MF fit
is the only slow step (about 15 seconds) -- and reuse them for every measurement below.

In [ ]:
import json
from collections import defaultdict

import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PopularityRetrievalPath,
    blend,
    generated_dir,
    intra_list_diversity,
    load_catalog,
    load_keywords,
    ndcg_at_k,
    novelty,
    precision_at_k,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")
keywords = load_keywords(data_dir / "keywords.csv.gz")
cold_readers = set(json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"])

popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = MatrixFactorizationPath(seed=0).fit(rows, catalog=catalog_ids)
paths = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}

print(f"catalog {len(catalog_ids)} books; {len(rows)} interaction rows")
print(f"cold readers held out of scoring: {len(cold_readers)}")
print(f"paths fitted: {', '.join(paths)}")

## 1. Ranking metrics: presence is not placement

Hit@k answers a yes/no question: *is at least one relevant book in the top k?* The moment it finds
one, it saturates at 1.0 -- it cannot tell a list that buries the right book at rank 10 from one
that leads with it, and it cannot reward a list that packs in *several* relevant books. Two
rank-position-aware metrics fix that.

- **precision@k** = (relevant books in the top k) / k. It rewards packing *more* of the top k with
  relevant items. The denominator is always k, so a short list after dropping already-seen books is
  still judged against the full budget.
- **NDCG@k** (normalized discounted cumulative gain) rewards putting the right book *high*. A
  relevant book at 1-based rank $r$ earns a gain of $1/\log_2(r+1)$, so rank 1 is worth 1.0, rank 2
  about 0.63, rank 3 about 0.50, and so on. We sum those gains (the DCG) and divide by the ideal DCG
  -- the score of the best possible ordering -- so NDCG lands in $[0, 1]$.

Compute both by hand on a tiny ranking, then confirm against `bookrec`.

In [ ]:
from math import log2

# A 4-book ranking; the reader's held-out relevant set is {2, 4}.
ranking = [1, 2, 4, 3]
relevant = {2, 4}

# precision@3 BY HAND: how many of the top 3 slots are relevant, over 3.
top3 = ranking[:3]
hand_precision = sum(1 for book in top3 if book in relevant) / 3
print(f"precision@3 by hand : {hand_precision:.4f}")
print(f"precision@3 bookrec : {precision_at_k(ranking, relevant, 3):.4f}")

# NDCG@5 BY HAND on [1,2,3,4,5] with relevant {2,5}: relevant books sit at ranks 2 and 5.
example = [1, 2, 3, 4, 5]
example_relevant = {2, 5}
dcg = 1 / log2(2 + 1) + 1 / log2(5 + 1)       # gains at ranks 2 and 5
idcg = 1 / log2(1 + 1) + 1 / log2(2 + 1)      # the ideal front-loads both relevant books
print(f"\nNDCG@5 by hand : {dcg / idcg:.4f}")
print(f"NDCG@5 bookrec : {ndcg_at_k(example, example_relevant, 5):.4f}")

### Why not RMSE?

If you have seen recommenders before, you may expect **RMSE** -- the root-mean-square error between
a predicted rating and the true rating. RMSE is the right tool for *rating prediction* ("guess this
user's 1-5 stars"), but it is the wrong tool here for two reasons. First, our feedback is
**implicit**: a read or a like is a 1, and everything else is *unknown*, not a 0 -- there are no
star ratings to be accurate about. Second, RMSE scores every prediction equally, while a top-k
recommender only ever shows the user a short list, so what matters is whether the *right books reach
the top of that list*. A model can have a lovely RMSE and still order the top 10 badly. Ranking
metrics -- hit@k, precision@k, NDCG@k -- score the thing the user actually sees.

### The deepened single-path scoreboard

The scoreboard already reports hit@10; precision@10 and NDCG@10 now ride alongside it (same readers,
same frozen `val` split, same cold-reader exclusion). Read the three columns together.

In [ ]:
print(f"{'path':>12} {'hit@10':>8} {'prec@10':>9} {'ndcg@10':>9}")
for name, path in paths.items():
    r = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )
    print(f"{name:>12} {r.hit_rate_at_k:>8.4f} {r.precision_at_k:>9.4f} {r.ndcg_at_k:>9.4f}")

The ladder's *order* is unchanged -- popularity < lexical < item-item < MF -- but the ranking
metrics open up the gaps the hit rate compressed. On NDCG@10, MF (about 0.106) pulls clearly ahead
of item-item CF (0.082): their hit rates were within sampling noise (0.276 vs 0.252), but MF places
relevant books *higher* in the list, which is exactly what a user feels. precision@10 stays low for
everyone (about 0.033 at best) -- natural when each reader has only a handful of held-out positives,
so even a perfect top 10 is mostly non-relevant slots. Precision and NDCG did not change the winner;
they changed how *decisively* MF wins, and that is information hit@10 threw away.

## 2. Beyond accuracy: coverage, diversity, novelty

Accuracy is not the whole job. A recommender also shapes *which* books get seen -- and two paths
with near-equal hit@k can serve wildly different catalogs. Three beyond-accuracy metrics make that
visible:

- **catalog coverage** -- the fraction of the whole catalog that appears in *some* reader's top k.
  Low coverage means the system funnels everyone toward the same few books.
- **intra-list diversity** -- `1 - mean pairwise similarity` within one reader's list; high means
  the list spans item space instead of ten near-duplicates.
- **novelty** -- the mean self-information $-\log_2 \hat{p}(\text{item})$ of the recommended books;
  rarer books score higher, so a path that only ever surfaces bestsellers has low novelty.

Measure coverage per path, then diversity and novelty on one warm reader's top 10.

In [ ]:
# Catalog coverage per single path (one-path blend: retrieve a pool of 30, rank the top 10).
print(f"{'path':>12} {'hit@10':>8} {'coverage':>10}")
for name, path in paths.items():
    b = run_blended_scoreboard(
        {name: path}, interactions_path, catalog_ids=catalog_ids,
        k=10, pool=30, weights={name: 1.0}, cold_readers=cold_readers,
    )
    print(f"{name:>12} {b.hit_rate_at_k:>8.4f} {b.catalog_coverage:>10.4f}")

# Diversity and novelty of one warm reader's top 10, with item-item CF supplying similarity.
cf_similarity = item_item.similarity
cf_row = {item: idx for idx, item in enumerate(item_item.artifact()["item_ids"])}


def item_similarity(a, b):
    if a in cf_row and b in cf_row:
        return float(cf_similarity[cf_row[a], cf_row[b]])
    return 0.0


seen_by_reader = defaultdict(set)
for row in rows:
    if row["split"] == "train" and row["label"] == 1:
        seen_by_reader[int(row["reader_id"])].add(int(row["item_id"]))

reader = int(mf.artifact()["reader_ids"][0])
seen = seen_by_reader[reader]
print(f"\nreader {reader} (history {len(seen)} books): one top-10, two beyond-accuracy lenses")
print(f"{'path':>12} {'diversity':>10} {'novelty':>9}")
for name, path in paths.items():
    recs = [c.item_id for c in path.retrieve(reader, {"seen": seen}, 10)]
    print(f"{name:>12} {intra_list_diversity(recs, item_similarity):>10.4f} "
          f"{novelty(recs, popularity.counts):>9.4f}")

The coverage column is the story. MF wins on accuracy yet covers only about **0.0745** of the
catalog -- roughly 150 of 2000 books ever reach anyone's top 10 -- while item-item CF reaches about
**0.49**, about 6.6x more. Popularity is the extreme: about **0.008** coverage, because it hands
*every* reader the same bestseller shelf (this is the **popularity bias** we met in Unit 2, now
measured directly). Lexical sits in between at about 0.28.

Novelty tells the same tale from the other side: popularity's list is the least novel (it is made of
the most-read books), while item-item CF surfaces rarer titles. None of this shows up in hit@10 --
two systems can tie on accuracy and serve completely different slices of the catalog. A recommender
that only optimizes hit@k quietly starves the long tail (Unit 4 already showed 818 books no reader
has co-read -- CF is blind to them, and popularity never reaches them either). Coverage, diversity,
and novelty are how we keep that cost in view.

## 3. Blend v1: combine the paths

Each path is strong somewhere and weak elsewhere -- MF is accurate but narrow, item-item CF is
broad, lexical catches text matches CF misses. The classic move is to **blend** them: retrieve a
pool of candidates from each path, merge the pools, and re-rank the union. The one subtlety is that
raw scores are not comparable -- MF returns dot products, popularity returns counts -- so `blend`
first **calibrates** each path's scores to $[0, 1]$ (min-max), then takes a **weighted** sum of the
calibrated scores per book, recording which paths proposed it. `rank` then drops already-seen books
and keeps the top k.

We measure at a **pinned** configuration -- per-path pool = 30, weights
`{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}` -- chosen in advance rather than searched.
`val` is the split we *select* on -- that is its job -- and `test` stays sealed for one final read
in Checkpoint A. But there is a discipline even within selection: do not report the single best
number from a `val` sweep as your expected performance, because selecting *and* reporting on the
same split biases the estimate upward. Pinning the knobs sidesteps that entirely. The pool is fixed
at 30 for honest reasons too -- a pool near k starves the blend, a bigger pool actually *lowers*
catalog coverage, and a bigger pool costs retrieval budget -- so 30 is a balance, not a number we
chased on `val`.

In [ ]:
weights = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
full = run_blended_scoreboard(
    paths, interactions_path, catalog_ids=catalog_ids,
    k=10, pool=30, weights=weights, cold_readers=cold_readers,
)
mf_only = run_blended_scoreboard(
    {"mf": mf}, interactions_path, catalog_ids=catalog_ids,
    k=10, pool=30, weights={"mf": 1.0}, cold_readers=cold_readers,
)
print("                  hit@10   coverage")
print(f"best single (MF)  {mf_only.hit_rate_at_k:.4f}   {mf_only.catalog_coverage:.4f}")
print(f"full blend        {full.hit_rate_at_k:.4f}   {full.catalog_coverage:.4f}")
print(f"\nblend vs MF: hit x{full.hit_rate_at_k / mf_only.hit_rate_at_k:.2f}, "
      f"coverage x{full.catalog_coverage / mf_only.catalog_coverage:.2f}")

Both wins are real: the blend reaches **hit@10 about 0.306**, past MF's 0.276 (the best single
path), *and* **coverage about 0.333** -- about 4.5x MF's 0.0745. Combining the paths lifts accuracy
a little and broadens reach a lot. That is the headline you would expect.

But "the blend beats its best part" invites a lazier claim: that *every* path is helping. Before we
believe it, we should test it -- the honest way to ask whether a component earns its place is to
remove it and see what happens.

### The leave-one-path-out ablation

For each path, drop it from the blend (keeping the other three at their pinned weights) and
re-measure hit@10. If a path contributes, removing it should *lower* the score.

In [ ]:
print(f"full blend: hit@10 {full.hit_rate_at_k:.4f}")
print(f"{'dropped path':>14} {'hit@10':>8} {'change':>8}")
for dropped in paths:
    kept = {name: path for name, path in paths.items() if name != dropped}
    kept_weights = {name: weights[name] for name in kept}
    b = run_blended_scoreboard(
        kept, interactions_path, catalog_ids=catalog_ids,
        k=10, pool=30, weights=kept_weights, cold_readers=cold_readers,
    )
    print(f"{dropped:>14} {b.hit_rate_at_k:>8.4f} {b.hit_rate_at_k - full.hit_rate_at_k:>+8.4f}")

Three of the four behave as expected: dropping **MF** costs the most (hit@10 falls to about 0.260,
a change of -0.046), then **item-item CF** (-0.026), then **lexical** (-0.014). Each of those three
clearly earns its place.

Then the surprise: dropping **popularity** *raises* hit@10, to about 0.308 (+0.002). Removing a
path makes the blend **better**. So "every path contributes" is simply **false** here -- popularity
has, if anything, slightly negative marginal value on accuracy. What establishes a path's
contribution is its **measured** leave-one-out change, not a story we tell in advance:
reader-dependence explains *why* a flat path like popularity is weak here, but it is not by itself
enough to guarantee positive marginal value -- only the measurement settles it. A path that
recommends the same thing to everyone can actively hurt a blend. Why would that be?

### Why popularity hurts: the calibration pitfall

The culprit is the very step that makes blending possible: **min-max calibration**. To put paths on
a common scale, `blend` maps each path's scores into $[0, 1]$ -- which forces **every path's top
candidate to exactly 1.0**. That is fine for a reader-dependent path, whose top candidate really is
*this* reader's best bet. But popularity is **reader-independent**: its top books are the same
bestsellers for everyone. After calibration, those generic bestsellers get a perfect 1.0 for *every*
reader, so with a non-trivial weight they tie -- and can outrank -- the reader's genuinely
personalized top pick, pushing a one-size-fits-all book into the top 10 where a relevant book should
be.

See the reader-independence directly: popularity returns the identical top list to two different
readers, while MF does not.

In [ ]:
warm_a, warm_b = (int(r) for r in mf.artifact()["reader_ids"][:2])
empty = {"seen": set()}
pop_a = [c.item_id for c in popularity.retrieve(warm_a, empty, 5)]
pop_b = [c.item_id for c in popularity.retrieve(warm_b, empty, 5)]
mf_a = [c.item_id for c in mf.retrieve(warm_a, empty, 5)]
mf_b = [c.item_id for c in mf.retrieve(warm_b, empty, 5)]
print(f"popularity top-5 for reader {warm_a}: {pop_a}")
print(f"popularity top-5 for reader {warm_b}: {pop_b}")
print(f"  -> identical across readers: {pop_a == pop_b}")
print(f"MF top-5 for reader {warm_a}: {mf_a}")
print(f"MF top-5 for reader {warm_b}: {mf_b}")
print(f"  -> identical across readers: {mf_a == mf_b}")

Popularity's lists match exactly; MF's differ. That is the whole mechanism. The pinned config does
*not* drop popularity to zero -- it keeps a small weight of 0.25 -- but **not** for accuracy or
coverage: its marginal accuracy is a wash (the ablation just showed removing it slightly *raises*
hit@10), and at this config dropping popularity actually *raises* catalog coverage too (0.333 ->
0.354). The one thing it buys is **cold-reader robustness**: it is the only path that can serve a
reader we know nothing about. Understanding *why* its accuracy contribution is flat is the real
lesson: calibrated blending lets a weak, flat path **tie** strong paths at the very top of the
pool, so adding paths is not free. Measure the ablation; do not assume.

### A robustness payoff: readers no personalized path can serve

That same flat popularity path buys something a pure-MF system cannot: **graceful degradation**.
Recall the 60 `cold_readers` held out of scoring -- readers with no training history. MF, item-item
CF, and lexical all need a reader's past to personalize, so for a cold reader they honestly return
nothing (`[]`). Popularity needs no history, so it -- and therefore the blend -- still produces a
sensible list. A blended system fails *softly*: when the personalized paths go silent, a
reader-independent path keeps the lights on.

In [ ]:
known_readers = set(mf.artifact()["reader_ids"])
cold_reader = next(r for r in sorted(cold_readers) if r not in known_readers)
print(f"cold reader {cold_reader} (no training history):")
for name, path in paths.items():
    served = len(path.retrieve(cold_reader, {"seen": set()}, 10))
    print(f"  {name:>12}: returns {served} books")

cold_pool = {name: path.retrieve(cold_reader, {"seen": set()}, 30) for name, path in paths.items()}
blended = rank(blend(cold_pool, weights=weights), n=10)
print(f"  {'blend':>12}: returns {len(blended)} books (popularity carries the cold reader)")

### A thread we are not closing yet: cold start

The cold *reader* above is one face of a deeper problem called **cold start** -- how to recommend
*to* a new reader, or *for* a brand-new book, when no path has any signal yet. Blending papers over
one corner of it (popularity can serve a new reader), but it cannot solve it: a book nobody has read
and no keyword distinguishes is invisible to every Part-1 path, popularity included. Genuinely
fixing cold start needs *content features* -- describing readers and books by their attributes, not
only their history -- which is where **Unit 9** picks the thread up, and its fairness consequences
are Unit 13's subject. For now, just notice the gap; we are not assessing it here.

## Where this goes

- **Ranking metrics** judge placement, not just presence: **precision@k** rewards packing the top k
  with relevant books, **NDCG@k** rewards putting them *high*. They widened MF's lead over CF that
  hit@10 had compressed. RMSE is for rating prediction, not implicit top-k.
- **Beyond-accuracy** metrics -- coverage, diversity, novelty -- reveal what accuracy hides: MF is
  accurate but covers about 0.0745 of the catalog, item-item CF reaches about 0.49, and popularity
  funnels everyone to about 0.008. Equal hit@k can mean very different catalogs served.
- **Blend v1** calibrates each path to $[0, 1]$, takes a weighted union, and ranks: hit@10 about
  0.306 (past MF's 0.276) **and** coverage about 0.333 (4.5x MF). The whole beats the best part.
- But the **leave-one-path-out ablation** refutes "every path helps": dropping MF, item-item, and
  lexical each hurts, while dropping **popularity slightly *raises*** hit@10. Min-max calibration
  ties a reader-independent path's bestsellers with every reader's true top pick, so its measured
  marginal value is about zero -- a path's contribution is settled by that measurement, not by
  whether it is reader-dependent. Popularity stays in the blend for **cold-reader robustness** alone
  (it carries the cold readers), not for accuracy or coverage -- dropping it actually *raises* both.

### The bridge ahead

Part 1 is complete: five retrieval paths and a blended system that is more than the sum of its
parts, measured on metrics that see accuracy *and* reach. **Checkpoint A** now assembles this
recommender and scores it once on the sealed `test` holdout -- the honest final grade for Part 1.
Then Part 2 begins learning the paths: Unit 8's neural two-tower, Unit 9's content features for cold
start, Unit 11's learned reranker that replaces the fixed-weight blend you just built with one whose
weights are *trained*.